In [0]:
# ============================================================
# NOTEBOOK   : silver_order_reviews
# PURPOSE    : Clean and transform bronze.order_reviews table
# SOURCE     : fincompliance_catalog.bronze.order_reviews
# DESTINATION: fincompliance_catalog.silver.order_reviews
# TRANSFORMATIONS:
#   - Deduplicate on review_id
#   - Cast review_score to IntegerType
#   - Handle massive nulls in comment columns
#   - Standardise review_score into sentiment
#   - Add has_comment flag
#   - Add review_response_days
#   - Drop bronze audit columns
#   - Add silver_updated_at audit column
# ============================================================

In [0]:

%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
# IMPORTS
# ============================================================
from pyspark.sql.functions import (
    col, trim, lower, when, lit,
    current_timestamp, datediff,
    count, avg, round as spark_round
)
from pyspark.sql.functions import try_to_timestamp
from pyspark.sql.types import IntegerType

In [0]:
## Read from Bronze
# ============================================================
df_order_reviews = spark.table(f"{BRONZE_DB}.order_reviews")
print(f"Total rows    : {df_order_reviews.count():,}")
print(f"Total columns : {len(df_order_reviews.columns)}")
df_order_reviews.show(3, truncate=True)

In [0]:
# DEDUPLICATION
# ============================================================
rows_before = df_order_reviews.count()
df_order_reviews_silver = df_order_reviews \
    .dropDuplicates(["review_id"])
rows_after = df_order_reviews_silver.count()
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Duplicates  : {rows_before - rows_after:,}")

In [0]:
# ============================================================
# CELL 6 - FIX REVIEW SCORE
# Keep only valid scores 1-5
# Cast to integer
# Set everything else to null
# ============================================================
df_order_reviews_silver = df_order_reviews_silver \
    .withColumn(
        "review_score",
        when(
            col("review_score").isin(
                ["1", "2", "3", "4", "5"]
            ),
            col("review_score").cast(IntegerType())
        )
        .otherwise(None)
    )

print("Review score distribution:")
df_order_reviews_silver \
    .groupBy("review_score") \
    .count() \
    .orderBy("review_score") \
    .show()

valid = df_order_reviews_silver \
    .filter(col("review_score").isNotNull()) \
    .count()
print(f"Valid scores   : {valid:,}")
print(f"Invalid scores : {df_order_reviews_silver.count() - valid:,}")

In [0]:
# ============================================================
# CAST DATE COLUMNS SAFELY
# Using try_to_timestamp which returns null
# instead of crashing on corrupted values
# ============================================================
df_order_reviews_silver = df_order_reviews_silver \
    .withColumn(
        "review_creation_date",
        try_to_timestamp(col("review_creation_date"))
    ) \
    .withColumn(
        "review_answer_timestamp",
        try_to_timestamp(col("review_answer_timestamp"))
    )

print("Date columns after safe casting:")
for field in df_order_reviews_silver.schema.fields:
    if field.name in [
        "review_creation_date",
        "review_answer_timestamp"
    ]:
        print(f"  {field.name} : {field.dataType}")

print("\nNull counts after casting:")
for c in ["review_creation_date", "review_answer_timestamp"]:
    null_count = df_order_reviews_silver \
        .filter(col(c).isNull()) \
        .count()
    print(f"  {c} : {null_count:,} nulls")

In [0]:
# HANDLE NULL COMMENT COLUMNS
# Replace nulls with empty string
# ============================================================
df_order_reviews_silver = df_order_reviews_silver \
    .withColumn(
        "review_comment_title",
        when(
            col("review_comment_title").isNull(),
            lit("")
        )
        .otherwise(trim(col("review_comment_title")))
    ) \
    .withColumn(
        "review_comment_message",
        when(
            col("review_comment_message").isNull(),
            lit("")
        )
        .otherwise(trim(col("review_comment_message")))
    )

print("Null counts after handling:")
for c in ["review_comment_title", "review_comment_message"]:
    null_count = df_order_reviews_silver \
        .filter(col(c).isNull()) \
        .count()
    print(f"  {c} : {null_count:,} nulls")

In [0]:
# ADD HAS COMMENT FLAG
# ============================================================
df_order_reviews_silver = df_order_reviews_silver \
    .withColumn(
        "has_comment",
        when(col("review_comment_message") != "", 1)
        .otherwise(0)
    )

print("Has comment breakdown:")
df_order_reviews_silver \
    .groupBy("has_comment") \
    .count() \
    .show()

In [0]:
# ADD SENTIMENT CLASSIFICATION
# ============================================================
df_order_reviews_silver = df_order_reviews_silver \
    .withColumn(
        "review_sentiment",
        when(col("review_score").isNull(), "unscored")
        .when(col("review_score").isin([1, 2]), "negative")
        .when(col("review_score") == 3, "neutral")
        .when(col("review_score").isin([4, 5]), "positive")
        .otherwise("unknown")
    )

print("Sentiment breakdown:")
df_order_reviews_silver \
    .groupBy("review_sentiment") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

In [0]:
# ADD REVIEW RESPONSE DAYS
# ============================================================
df_order_reviews_silver = df_order_reviews_silver \
    .withColumn(
        "review_response_days",
        when(
            col("review_answer_timestamp").isNull() |
            col("review_creation_date").isNull(),
            None
        )
        .otherwise(
            datediff(
                col("review_answer_timestamp"),
                col("review_creation_date")
            )
        )
    )

print("Review response days statistics:")
df_order_reviews_silver \
    .filter(col("review_response_days").isNotNull()) \
    .select("review_response_days") \
    .summary("min", "max", "mean") \
    .show()

In [0]:
# CELL 12 - DROP BRONZE AUDIT COLUMNS
# ============================================================
df_order_reviews_silver = df_order_reviews_silver \
    .drop("ingestion_timestamp", "source_file")

print("Columns after dropping audit columns:")
for col_name in df_order_reviews_silver.columns:
    print(f"  {col_name}")

In [0]:
# ADD SILVER AUDIT COLUMN
# ============================================================
df_order_reviews_silver = df_order_reviews_silver \
    .withColumn("silver_updated_at", current_timestamp())

print(f"Total columns : {len(df_order_reviews_silver.columns)}")
print(f"Total rows    : {df_order_reviews_silver.count():,}")

In [0]:
# WRITE TO SILVER
# ============================================================
(
    df_order_reviews_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.order_reviews")
)

print(f"Written to {SILVER_DB}.order_reviews")
print(f"Rows    : {df_order_reviews_silver.count():,}")
print(f"Columns : {len(df_order_reviews_silver.columns)}")

In [0]:
# VERIFICATION
# ============================================================
print("=" * 60)
print("SILVER.ORDER_REVIEWS VERIFICATION")
print("=" * 60)

df_silver = spark.table(f"{SILVER_DB}.order_reviews")

bronze_count = spark.table(
    f"{BRONZE_DB}.order_reviews").count()
silver_count = df_silver.count()

print(f"\nCHECK 1 - Row counts:")
print(f"  Bronze : {bronze_count:,}")
print(f"  Silver : {silver_count:,}")
if bronze_count == silver_count:
    print(f"  Row counts match")
else:
    print(f"  Difference : {bronze_count - silver_count:,}")

print(f"\nCHECK 2 - New columns:")
new_cols = [
    "has_comment",
    "review_sentiment",
    "review_response_days",
    "silver_updated_at"
]
for c in new_cols:
    if c in df_silver.columns:
        print(f"  {c} exists")
    else:
        print(f"  {c} MISSING")

print(f"\nCHECK 3 - Sentiment breakdown:")
df_silver \
    .groupBy("review_sentiment") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

print(f"\nCHECK 4 - Review score distribution:")
df_silver \
    .groupBy("review_score") \
    .count() \
    .orderBy("review_score") \
    .show()

print("=" * 60)
print("silver.order_reviews verification complete!")
print(f"Rows  : {silver_count:,}")
print(f"Cols  : {len(df_silver.columns)}")
print("=" * 60)